# Día 5 — Entropía (cuando la probabilidad no basta)

## Objetivo del notebook
Aprender a medir **incertidumbre** en las predicciones del modelo cuando
mirar solo la probabilidad “ganadora” (**confidence**) no es suficiente
para tomar una decisión confiable.

En problemas reales —especialmente **multiclase**— un modelo puede mostrar
una probabilidad alta y aun así estar **confundido** entre varias opciones.
Este notebook introduce la **entropía** como señal complementaria para
detectar esos casos.

---

## Probabilidad (recordatorio rápido)
Una probabilidad es un número entre **0 y 1** que indica qué tan posible es
que ocurra un evento.

Cuando un modelo predice, no solo da una decisión final, sino un conjunto
de probabilidades que reflejan cómo reparte su “creencia” entre las
distintas clases posibles.

---

## Clases y predicciones en multiclase
En un problema multiclase, el modelo asigna **una probabilidad a cada clase**.
Por ejemplo:

- Clase A → 0.55  
- Clase B → 0.40  
- Clase C → 0.05  

El modelo **elige la clase con mayor probabilidad**, pero eso no implica que
la decisión sea clara.

---

## Confidence (probabilidad top-1)
**Confidence** es la **probabilidad más alta** entre todas las clases.
También se conoce como **probabilidad top-1**.

Representa:
> qué tan fuerte es la opción que el modelo considera más probable.

Sin embargo, confidence **no muestra** qué tan cerca estuvo la segunda mejor
opción.

---

## Confusión entre clases
Un `confidence` alto puede ser engañoso cuando la **segunda clase más
probable** tiene una probabilidad cercana.

Ejemplo:
- Clase A → 0.55  
- Clase B → 0.45  

Aunque el modelo “elige” A, está casi igual de inclinado por B.
Esta **confusión entre la 1ª y la 2ª clase** no se ve mirando solo
`confidence`.

---

## Entropía: incertidumbre informativa
La **entropía** mide **qué tan repartida está la probabilidad entre las
clases**.

- **Entropía baja**  
  La mayoría de la probabilidad está concentrada en una sola clase.  
  El modelo tiene una preferencia clara.

- **Entropía alta**  
  La probabilidad está distribuida entre varias clases.  
  El modelo duda entre alternativas plausibles.

La entropía **no reemplaza** a la probabilidad.
La complementa.

---

## Idea clave del Día 5
- **Confidence** responde: *¿qué clase gana?*  
- **Entropía** responde: *¿qué tan clara fue esa victoria?*

Este notebook usa entropía (y luego variabilidad) para identificar
**casos engañosos** y definir una **zona gris** donde la predicción requiere
intervención adicional.

---

### **Variabilidad**

**Concepto simple**
La **variabilidad** mide **qué tanto cambia la predicción del modelo para el mismo caso**
cuando repetimos la inferencia de formas ligeramente distintas.

No mira *qué clase ganó*.
Mira **qué tan estable es esa predicción**.

---

### **Ejemplo claro**

Mismo registro, cinco modelos similares (ensemble):

* Modelo 1 → 0.81
* Modelo 2 → 0.79
* Modelo 3 → 0.80
* Modelo 4 → 0.82
* Modelo 5 → 0.78

👉 **Baja variabilidad**
El modelo es consistente → mayor certeza.

---

Otro caso, mismo registro:

* Modelo 1 → 0.30
* Modelo 2 → 0.65
* Modelo 3 → 0.55
* Modelo 4 → 0.40
* Modelo 5 → 0.75

👉 **Alta variabilidad**
El modelo cambia de opinión → **incertidumbre**, aunque el promedio sea alto.

---

### **Traducción directa a negocio**

* **Baja variabilidad** → decisión estable, automatizable.
* **Alta variabilidad** → riesgo oculto, requiere intervención.

---

### **Idea clave**

> La variabilidad responde:
> **¿El modelo se mantiene firme o duda cuando lo miro varias veces?**

Eso es exactamente lo que vamos a comparar con **confidence** y **entropía** en este notebook.


---
---
---

## Qué vamos a hacer en este notebook (Día 5)

Vamos a construir **tres señales** para medir incertidumbre y compararlas en los mismos casos:

1) **Confidence (top-1)**  
   La probabilidad más alta. Es lo que normalmente se usa para decidir.

2) **Entropía (softmax)**  
   Mide si la probabilidad está **concentrada** en una clase (decisión clara) o **repartida** entre varias (confusión).  
   Es especialmente útil en **multiclase**.

3) **Variabilidad (ensemble)**  
   Mide qué tan **estable** es la predicción para el mismo caso cuando repetimos el modelo (distintas semillas / subconjuntos).  
   Variabilidad alta = riesgo de fragilidad.

Con estas señales vamos a:
- Identificar **casos engañosos** (ej. confidence alto pero entropía/variabilidad alta).
- Definir una **zona gris** para intervención.
- Preparar la **matriz de decisión** con 2 señales (p vs entropía o p vs variabilidad).


In [1]:
# CELDA 1 — Imports y configuración
import numpy as np
import pandas as pd

EPS = 1e-12

In [2]:
# CELDA 2 — Funciones de señales (confidence, entropía, entropía normalizada, variabilidad)

def confidence_top1(probs_row):
    """Probabilidad top-1 (confidence) en multiclase."""
    return float(np.max(probs_row))

def entropy_softmax(probs_row):
    """Entropía: mayor = probabilidad más repartida (más incertidumbre)."""
    p = np.clip(np.array(probs_row, dtype=float), EPS, 1.0)
    p = p / p.sum()
    return float(-np.sum(p * np.log(p)))

def entropy_normalized(probs_row):
    """Entropía normalizada a [0,1] para comparar entre distintos K."""
    p = np.array(probs_row, dtype=float)
    K = len(p)
    H = entropy_softmax(p)
    return float(H / np.log(K))

def variability_std(ensemble_probs):
    """
    Variabilidad: desviación estándar del p_max entre modelos del ensemble.
    ensemble_probs: (M, K)
    """
    pmax = np.max(ensemble_probs, axis=1)  # p_max por modelo
    return float(np.std(pmax, ddof=1)) if len(pmax) > 1 else 0.0


In [3]:
# CELDA 3 — Datos sintéticos (demo) para 6 casos, 3 clases, 5 modelos por caso

cases = [
    # Caso 0: decisión clara y estable
    np.array([
        [0.92, 0.06, 0.02],
        [0.90, 0.07, 0.03],
        [0.93, 0.05, 0.02],
        [0.91, 0.06, 0.03],
        [0.92, 0.05, 0.03],
    ]),
    # Caso 1: confidence alto pero confusión con 2ª clase
    np.array([
        [0.65, 0.30, 0.05],
        [0.62, 0.33, 0.05],
        [0.66, 0.28, 0.06],
        [0.64, 0.31, 0.05],
        [0.63, 0.32, 0.05],
    ]),
    # Caso 2: distribución repartida (entropía alta) y estable
    np.array([
        [0.38, 0.34, 0.28],
        [0.36, 0.35, 0.29],
        [0.39, 0.33, 0.28],
        [0.37, 0.34, 0.29],
        [0.38, 0.33, 0.29],
    ]),
    # Caso 3: confidence alto promedio pero inestable (variabilidad alta)
    np.array([
        [0.95, 0.03, 0.02],
        [0.70, 0.20, 0.10],
        [0.85, 0.10, 0.05],
        [0.60, 0.30, 0.10],
        [0.92, 0.05, 0.03],
    ]),
    # Caso 4: top-2 muy cerca (entropía medio-alta) y algo inestable
    np.array([
        [0.52, 0.46, 0.02],
        [0.55, 0.43, 0.02],
        [0.50, 0.48, 0.02],
        [0.58, 0.40, 0.02],
        [0.54, 0.44, 0.02],
    ]),
    # Caso 5: low confidence pero estable
    np.array([
        [0.44, 0.41, 0.15],
        [0.43, 0.42, 0.15],
        [0.45, 0.40, 0.15],
        [0.44, 0.40, 0.16],
        [0.43, 0.41, 0.16],
    ]),
]


In [4]:
# CELDA 4 — Calcular señales por caso y armar tabla

rows = []
for i, ens in enumerate(cases):
    mean_probs = ens.mean(axis=0)  # promedio de probs por clase (agregación simple)
    pmax = confidence_top1(mean_probs)
    H = entropy_softmax(mean_probs)
    Hn = entropy_normalized(mean_probs)
    std_pmax = variability_std(ens)

    # 2ª clase = segunda prob más alta en el promedio
    second = float(np.sort(mean_probs)[-2])

    rows.append({
        "case_id": i,
        "mean_probs": np.round(mean_probs, 3),
        "p_max (confidence)": round(pmax, 3),
        "p_2nd": round(second, 3),
        "entropy": round(H, 3),
        "entropy_norm": round(Hn, 3),
        "std_pmax (variability)": round(std_pmax, 3),
    })

df = pd.DataFrame(rows)
df_sorted = df.sort_values(by=["entropy_norm", "std_pmax (variability)"], ascending=False).reset_index(drop=True)

df_sorted


,case_id,mean_probs,p_max (confidence),p_2nd,entropy,entropy_norm,std_pmax (variability)
0,2,"[0.376, 0.338, 0.286]",0.376,0.338,1.092,0.994,0.011
1,5,"[0.438, 0.408, 0.154]",0.438,0.408,1.015,0.924,0.008
2,1,"[0.64, 0.308, 0.052]",0.640,0.308,0.802,0.730,0.016
3,4,"[0.538, 0.442, 0.02]",0.538,0.442,0.773,0.703,0.030
4,3,"[0.804, 0.136, 0.06]",0.804,0.136,0.616,0.560,0.149
5,0,"[0.916, 0.058, 0.026]",0.916,0.058,0.340,0.310,0.011


In [5]:
# CELDA 5 — Regla preliminar de "zona gris" (ajustable con data real)

p_hi = 0.80     # umbral de confidence alto
Hn_hi = 0.65    # umbral de entropía alta (normalizada)
std_hi = 0.08   # umbral de variabilidad alta

def decision_rule(pmax, Hn, std):
    if pmax >= p_hi and Hn < Hn_hi and std < std_hi:
        return "AUTO (alto control)"
    if pmax >= p_hi and (Hn >= Hn_hi or std >= std_hi):
        return "ZONA GRIS (revisión/2ª validación)"
    if pmax < p_hi and (Hn >= Hn_hi or std >= std_hi):
        return "ESCALAR (alta incertidumbre)"
    return "GUIAR (reglas/contexto)"

df_sorted["action (draft)"] = df_sorted.apply(
    lambda r: decision_rule(r["p_max (confidence)"], r["entropy_norm"], r["std_pmax (variability)"]),
    axis=1
)

df_sorted[["case_id", "p_max (confidence)", "p_2nd", "entropy_norm", "std_pmax (variability)", "action (draft)"]]


,case_id,p_max (confidence),p_2nd,entropy_norm,std_pmax (variability),action (draft)
0,2,0.376,0.338,0.994,0.011,ESCALAR (alta incertidumbre)
1,5,0.438,0.408,0.924,0.008,ESCALAR (alta incertidumbre)
2,1,0.640,0.308,0.730,0.016,ESCALAR (alta incertidumbre)
3,4,0.538,0.442,0.703,0.030,ESCALAR (alta incertidumbre)
4,3,0.804,0.136,0.560,0.149,ZONA GRIS (revisión/2ª validación)
5,0,0.916,0.058,0.310,0.011,AUTO (alto control)



---

## Cómo leer la tabla (reglas mentales rápidas)

* **p_max (confidence)**
  Qué tan fuerte “gana” la clase elegida.

* **p_2nd**
  Qué tan cerca estuvo la segunda clase.
  Cercana a `p_max` ⇒ **confusión**.

* **entropy_norm**
  Qué tan repartida está la probabilidad.
  Cerca de **1** ⇒ mucha duda.
  Cerca de **0** ⇒ decisión clara.

* **std_pmax (variability)**
  Qué tan estable es la predicción entre modelos.
  Alta ⇒ el modelo **no es firme**.

* **action**
  Política preliminar de decisión.

---

## Análisis por casos

### **Caso 0 (case_id = 2)**

* p_max = 0.376 (bajo)
* p_2nd = 0.338 (muy cerca)
* entropía ≈ 1.0 (máxima)
* variabilidad baja

**Lectura:**
El modelo **no sabe qué clase elegir**.
No hay señal dominante.

**Decisión:**
👉 **ESCALAR**
Este es el ejemplo puro de **incertidumbre informativa** (entropía).

---

### **Caso 1 (case_id = 5)**

* p_max = 0.438
* p_2nd = 0.408
* entropía muy alta
* variabilidad baja

**Lectura:**
El modelo es estable, pero **confundido**.
Siempre duda entre las mismas clases.

**Decisión:**
👉 **ESCALAR**
Caso ambiguo por naturaleza del dato.

---

### **Caso 2 (case_id = 1)**

* p_max = 0.640
* p_2nd = 0.308
* entropía alta (0.73)
* variabilidad baja

**Lectura:**
Hay una clase ganadora, pero el resto **no es despreciable**.
La confianza no es suficiente para automatizar.

**Decisión:**
👉 **ESCALAR**
Buen ejemplo de *“probabilidad media no implica certeza”*.

---

### **Caso 3 (case_id = 4)**

* p_max = 0.538
* p_2nd = 0.442 (muy cerca)
* entropía alta
* variabilidad moderada

**Lectura:**
Competencia directa entre top-1 y top-2.
Este es el clásico **caso engañoso** si solo miras `p_max`.

**Decisión:**
👉 **ESCALAR**

---

### **Caso 4 (case_id = 3)**

* p_max = 0.804 (alto)
* p_2nd = 0.136 (lejos)
* entropía moderada
* **variabilidad muy alta (0.149)**

**Lectura clave:**
Aquí está uno de los **hallazgos más importantes del notebook**.

* La probabilidad promedio es alta.
* La entropía no es alarmante.
* **Pero el modelo no es estable**: cambia mucho entre entrenamientos.

**Interpretación:**
No es confusión semántica.
Es **fragilidad del modelo**.

**Decisión:**
👉 **ZONA GRIS**
Revisión, segundo modelo, o features adicionales.

---

### **Caso 5 (case_id = 0)**

* p_max = 0.916
* p_2nd = 0.058
* entropía baja
* variabilidad baja

**Lectura:**
Caso ideal:

* Clase dominante clara.
* Distribución concentrada.
* Predicción estable.

**Decisión:**
👉 **AUTO (alto control)**
Este es el **gold standard** para automatización.

---

## Conclusiones clave del Día 5

### 1. `p_max` **no alcanza**

Casos 1, 3 y 4 muestran que una probabilidad “decente” puede ocultar:

* confusión entre clases (entropía),
* o fragilidad del modelo (variabilidad).

---

### 2. Entropía y variabilidad **detectan riesgos distintos**

* **Entropía alta** → ambigüedad del dato.
* **Variabilidad alta** → debilidad del modelo.

Ambas son críticas y **no se sustituyen**.

---

### 3. La “zona gris” es real y medible

No es intuición.
Es una región definida por señales cuantificables.

---

### 4. Resultado estratégico

Con este notebook ya no decides solo con:

> “el modelo dice 0.8”

Decides con:

> “dice 0.8, es estable, no está confundido → puedo automatizar”

---